In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import random
import copy
import time

# ==========================================
# 1. NETWORK ARCHITECTURE (Same size as KFN)
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                                          nn.BatchNorm2d(out_c))
    def forward(self, x):
        return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.final_conv = nn.Conv2d(256, 64, 1)
    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = [ResidualBlock(in_c, out_c, stride)]
        for _ in range(1, blocks): layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)
    def forward(self, x):
        return self.final_conv(self.pool(self.layer3(self.layer2(self.layer1(self.relu(self.bn1(self.conv1(x))))))))

class BaselineNet(nn.Module):
    def __init__(self, n_classes=10, n_specs=2):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        self.classifier = nn.Linear(64, n_classes)
    def forward(self, x):
        feat = sum([s(x) for s in self.specialists]) / len(self.specialists)
        feat = feat.mean(dim=(2,3))  # global avg pool
        return self.classifier(feat)

# ==========================================
# 2. UTILITIES
# ==========================================
def set_deterministic(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

def get_dataset(transform_train, transform_test):
    root = '/kaggle/input/cifar-10'
    train_ds = torchvision.datasets.CIFAR10(root=root, train=True, download=False, transform=transform_train)
    test_ds = torchvision.datasets.CIFAR10(root=root, train=False, download=False, transform=transform_test)
    return train_ds, test_ds

# ==========================================
# 3. LwF TRAINING
# ==========================================
def run_lwf_baseline(seed, loaders, epochs_A, epochs_B, device, alpha=1.0, T=2.0):
    set_deterministic(seed)
    l_A, l_B, t_A, t_B, replay_loader = loaders

    scaler = torch.amp.GradScaler('cuda') if getattr(device,"type","")=="cuda" else None

    # --- Model Initialization ---
    model = BaselineNet(n_classes=10, n_specs=2).to(device)
    teacher = copy.deepcopy(model).eval()

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # --- Phase 1: Task A Training ---
    for _ in range(epochs_A):
        model.train()
        for x, y in l_A:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            if scaler:
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                opt.step()

    acc_A_init = evaluate(model, t_A, device)
    teacher.load_state_dict(model.state_dict())  # Task A teacher

    # --- Phase 2: Task B Training with LwF ---
    rep_iter = iter(replay_loader)
    for _ in range(epochs_B):
        model.train()
        for x_B, y_B in l_B:
            x_B, y_B = x_B.to(device), y_B.to(device)
            try:
                x_A, y_A = next(rep_iter)
            except StopIteration:
                rep_iter = iter(replay_loader)
                x_A, y_A = next(rep_iter)
            x_A, y_A = x_A.to(device), y_A.to(device)

            opt.zero_grad()

            # Task A distillation
            logits_A_student = model(x_A)
            with torch.no_grad():
                logits_A_teacher = teacher(x_A)
            l_distill = F.kl_div(F.log_softmax(logits_A_student/T, dim=1),
                                 F.softmax(logits_A_teacher/T, dim=1),
                                 reduction='batchmean') * (T**2)

            # Task B loss
            logits_B = model(x_B)
            l_taskB = F.cross_entropy(logits_B, y_B)

            loss = l_taskB + alpha * l_distill

            if scaler:
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                opt.step()

    # --- Evaluation ---
    acc_A_final = evaluate(model, t_A, device)
    acc_B_final = evaluate(model, t_B, device)

    return {"acc_A_init": acc_A_init, "acc_A_final": acc_A_final, "acc_B_final": acc_B_final}

# ==========================================
# 4. RUN ALL SEEDS & REPORT
# ==========================================
def run_all():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    stats = ((0.4914,0.4822,0.4465),(0.247,0.243,0.261))
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.2),
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])
    t_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])

    train_ds, test_ds = get_dataset(t_train, t_test)

    EPOCHS_A = 10
    EPOCHS_B = 40
    REPLAY_SIZE = 12000
    BS = 256
    NW = 2

    loaders = (
        DataLoader(Subset(train_ds, [i for i,l in enumerate(train_ds.targets) if l<5]), BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True),
        DataLoader(Subset(train_ds, [i for i,l in enumerate(train_ds.targets) if l>=5]), BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True),
        DataLoader(Subset(test_ds, [i for i,l in enumerate(test_ds.targets) if l<5]), BS, num_workers=NW, pin_memory=True),
        DataLoader(Subset(test_ds, [i for i,l in enumerate(test_ds.targets) if l>=5]), BS, num_workers=NW, pin_memory=True),
        DataLoader(Subset(train_ds, [i for i,l in enumerate(train_ds.targets) if l<5][:REPLAY_SIZE]), BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True)
    )

    seeds = [0,1,2,3,4]
    results = []
    for s in seeds:
        print(f" -> Running LwF Seed {s}...")
        r = run_lwf_baseline(s, loaders, EPOCHS_A, EPOCHS_B, device)
        results.append(r)
        print(f"    ↳ Seed {s} Results: Task A Init={r['acc_A_init']:.2f} | Task A Final={r['acc_A_final']:.2f} | Task B Final={r['acc_B_final']:.2f}")

    print("\n=== LwF Baseline (Mean ± Std over 5 Seeds) ===")
    for k, label in [("acc_A_init","Task A Init"),("acc_A_final","Task A Final"),("acc_B_final","Task B Final")]:
        vals = [r[k] for r in results]
        print(f"{label:15}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

if __name__ == "__main__":
    run_all()

 -> Running LwF Seed 0...
    ↳ Seed 0 Results: Task A Init=75.00 | Task A Final=74.10 | Task B Final=32.24
 -> Running LwF Seed 1...
    ↳ Seed 1 Results: Task A Init=80.26 | Task A Final=80.02 | Task B Final=21.84
 -> Running LwF Seed 2...
    ↳ Seed 2 Results: Task A Init=82.02 | Task A Final=80.26 | Task B Final=35.56
 -> Running LwF Seed 3...
    ↳ Seed 3 Results: Task A Init=81.12 | Task A Final=78.56 | Task B Final=37.12
 -> Running LwF Seed 4...
    ↳ Seed 4 Results: Task A Init=81.52 | Task A Final=79.86 | Task B Final=33.60

=== LwF Baseline (Mean ± Std over 5 Seeds) ===
Task A Init    : 79.98 ± 2.56
Task A Final   : 78.56 ± 2.31
Task B Final   : 32.07 ± 5.38
